In [1]:
# Colab Setup - Run this cell first if using Google Colab
# Skip this cell if running locally with torchref already installed

#install torchref
!pip install torchref

# Download a structure/dataset pair 
!wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.pdb
!wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.mtz

mtz = "./1DAW.mtz"
pdb = "./1DAW.pdb"

# Structure Factor Calculation in TorchRef

This notebook demonstrates the different levels of abstraction available for computing structure factors from atomic models. TorchRef provides four approaches, from fully automatic to fully manual, allowing you to choose the right level of control for your use case.

## Overview of Approaches

1. **Fully Automatic** - One-liner using `ModelFT.__call__()` - best for standard refinement
2. **Semi-Automatic** - Using the `FFT` class directly - useful when you need the electron density map
3. **Intermediate** - Building density map and extracting structure factors separately
4. **Fully Manual** - Direct access to all underlying functions - for custom workflows or debugging

## Imports and Setup

In [2]:
# Core imports:
# - ModelFT: The main model class that handles atomic coordinates and computes structure factors
# - ReflectionData: Handles experimental reflection data (F_obs, sigmas, R-free flags)
# - dtypes: Configuration for float precision (float32 vs float64)

from torchref import ModelFT, ReflectionData
from torchref.config import dtypes

/tmp/ipykernel_2099915/996263503.py:6: UserWarning: TorchRef auto-configured 4 threads. Set TORCHREF_NUM_THREADS to override.
  from torchref import ModelFT, ReflectionData


In [3]:
import os

# File paths - use Colab-downloaded files if available, else use local paths
if os.path.exists('./1DAW.mtz'):
    # Running in Colab with downloaded files
    mtz = './1DAW.mtz'
    pdb = './1DAW.pdb'
else:
    # Running locally - update these paths to your data files
    mtz = '/das/work/p17/p17490/Peter/Library/torchref/tests/files/mtz/1DAW.mtz'
    pdb = '/das/work/p17/p17490/Peter/Library/torchref/tests/files/pdb/1DAW.pdb'

# Load the atomic model
# max_res sets the resolution limit which determines:
#   - FFT grid sampling (Shannon-Nyquist criterion)
#   - Which reflections are included in calculations
m = ModelFT(max_res=2.0).load_pdb(pdb)

# Load reflection data
# This handles:
#   - Reading HKL indices, amplitudes/intensities, sigmas
#   - French-Wilson treatment if intensities are provided
#   - R-free flag extraction
data = ReflectionData().load_mtz(mtz)

Loaded 3051 atoms
FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


## Loading Data

Load the atomic model (PDB) and experimental reflection data (MTZ). 

**Key parameters:**
- `max_res`: Maximum resolution in Ångström - determines the FFT grid size
- The model automatically extracts atom types, positions, B-factors, and occupancies
- ReflectionData applies French-Wilson treatment to convert intensities to amplitudes if needed

In [4]:
# Extract experimental data from ReflectionData object
# Returns: hkl indices, observed amplitudes, sigmas, and R-free flags
hkl, F_obs, sig_F_obs, rfree = data()

# Compute structure factors - just one line!
# The model is callable: m(hkl) computes F_calc for the given Miller indices
# Returns complex tensor with amplitude and phase information
f_calc_automatic = m(hkl)

Parametrization built for 6 unique atom types


## Approach 1: Fully Automatic

The simplest way to compute structure factors. Just call the model with HKL indices.

**When to use:** Standard refinement workflows where you don't need intermediate results.

**What happens internally:**
1. Builds electron density map on a 3D grid
2. Applies FFT to get reciprocal space
3. Extracts structure factors at requested HKL positions with symmetry averaging

In [5]:
# Get crystallographic information from the model
cell = m.cell           # Unit cell with lattice parameters and matrices
spacegroup = m.spacegroup  # Space group with symmetry operations

from torchref.model import FFT

# Get isotropic atom parameters as a tuple:
# - xyz_iso: fractional coordinates (N, 3)
# - adp_iso: B-factors (N,)
# - occ_iso: occupancies (N,)
# - A_iso, B_iso: Gaussian scattering factor coefficients
parameters = m.get_iso()

# Get anisotropic atom parameters (if any exist in the model)
# For isotropic-only models, these tensors will be empty
parameter_aniso = m.get_aniso()

# Create FFT calculator with cell geometry and resolution
SF_calc = FFT(cell=cell, spacegroup=spacegroup, max_res=2.0)

# Option A: Build density map only (useful for visualization or custom processing)
# apply_symmetry=False means only ASU atoms contribute; set True to expand to full cell
Electron_density = SF_calc.build_density_map(
    *parameters, *parameter_aniso, 
    apply_symmetry=False
)

# Option B: Compute structure factors in one step (builds density internally)
# Returns both F_calc and the density map
f_calc_semi_automatic, Electron_density = SF_calc.compute_structure_factors(
    hkl, *parameters, *parameter_aniso
)

# Option C: Extract structure factors from an existing density map
# Useful when you've modified the density or want different HKL sets
f_calc_a_bit_less_automatic = SF_calc.map_to_structure_factors(
    Electron_density, hkl, 
    apply_symmetry=True  # Average over symmetry-equivalent reflections
)

## Approach 2: Semi-Automatic (via FFT class)

Use the `FFT` class directly for more control over the calculation.

**When to use:** 
- When you need access to the electron density map
- When computing structure factors for multiple HKL sets from the same density
- When you want to apply symmetry at different stages

**Key concepts:**
- **Isotropic parameters:** Position, single B-factor, occupancy, scattering factors (A, B coefficients)
- **Anisotropic parameters:** Position, 6-element ADP tensor (U11, U22, U33, U12, U13, U23), occupancy, scattering factors

In [6]:
# Get crystallographic setup
cell = m.cell
spacegroup = m.spacegroup

# Import low-level functions from the base module
from torchref.base import (
    get_real_grid,                          # Generate fractional coordinate grid
    vectorized_add_to_map,                  # Add isotropic atom contributions
    vectorized_add_to_map_aniso,            # Add anisotropic atom contributions  
    find_relevant_voxels,                   # Find voxels within atom's influence radius
    ifft,                                   # Inverse FFT (real -> reciprocal space)
    extract_structure_factors_with_symmetry # Sample F at HKL with symmetry
)
import torch

# ============================================================================
# Step 1: Set up the real-space grid
# ============================================================================
# Grid size is determined by resolution to satisfy Shannon-Nyquist sampling
# Rule of thumb: grid spacing ≈ resolution / 3
gridsize = cell.compute_grid_size(max_res=2.0)

# Create fractional coordinate grid (values 0 to 1 in each dimension)
fractional_grid = get_real_grid(
    fractional_matrix=cell.fractional_matrix, 
    gridsize=gridsize
)

# Initialize empty density map
Electron_density_map = torch.zeros(gridsize, dtype=dtypes.float)

# ============================================================================
# Step 2: Unpack atomic parameters
# ============================================================================
xyz_iso, adp_iso, occ_iso, A_iso, B_iso = parameters
xyz_aniso, adp_aniso, occ_aniso, A_aniso, B_aniso = parameter_aniso

# ============================================================================
# Step 3: Build isotropic density contributions
# ============================================================================
# For each atom, find which voxels are within its influence radius
# This avoids computing contributions at distant voxels (significant speedup)
surrounding_coords, voxel_indices = find_relevant_voxels(
    fractional_grid,
    xyz_iso,
    radius_angstrom=3.0,  # Cutoff radius - larger = more accurate but slower
    inv_frac_matrix=cell.inv_fractional_matrix,
)

# Add Gaussian density contributions from all isotropic atoms
# Uses the 4-Gaussian approximation to atomic scattering factors
density_map = vectorized_add_to_map(
    surrounding_coords,    # Voxel positions near each atom
    voxel_indices,         # Which voxels to update
    Electron_density_map,  # Map to add to (modified in place)
    xyz_iso,               # Atom positions (fractional)
    adp_iso,               # B-factors
    cell.inv_fractional_matrix,
    cell.fractional_matrix,
    A_iso,                 # Scattering factor A coefficients
    B_iso,                 # Scattering factor B coefficients
    occ_iso,               # Occupancies
)

# ============================================================================
# Step 4: Add anisotropic density contributions (if any)
# ============================================================================
if xyz_aniso.numel() > 0:  # Check if there are anisotropic atoms
    relevant_coords_aniso, voxel_indices_aniso = find_relevant_voxels(
        fractional_grid,
        xyz_aniso,
        radius_angstrom=3.0,
        inv_frac_matrix=cell.inv_fractional_matrix,
    )
    # Anisotropic version uses full U tensor instead of scalar B
    density_map = vectorized_add_to_map_aniso(
        relevant_coords_aniso,
        voxel_indices_aniso,
        density_map,
        xyz_aniso,
        adp_aniso,  # Shape (N, 6) for U11, U22, U33, U12, U13, U23
        cell.inv_fractional_matrix,
        cell.fractional_matrix,
        A_aniso,
        B_aniso,
        occ_aniso,
    )

# ============================================================================
# Step 5: FFT to reciprocal space
# ============================================================================
# Transform density map to reciprocal space
# Normalization by cell volume gives structure factors in electrons
reciprocal_grid = ifft(density_map)

# ============================================================================
# Step 6: Extract structure factors at HKL positions
# ============================================================================
# Sample the reciprocal space grid at integer HKL positions
# Applies symmetry averaging using space group operations
f_calc_manual = extract_structure_factors_with_symmetry(
    reciprocal_grid,
    hkl,
    spacegroup.matrices,      # Rotation matrices for symops
    spacegroup.translations   # Translation vectors for symops
)

## Approach 3: Fully Manual

Direct access to all underlying functions for maximum control.

**When to use:**
- Custom electron density modifications
- Debugging or understanding the algorithm
- Research applications requiring non-standard workflows
- Performance optimization for specific use cases

**The calculation pipeline:**
1. **Grid setup:** Determine grid size from resolution (Shannon-Nyquist)
2. **Voxel finding:** Identify which voxels each atom contributes to (within cutoff radius)
3. **Density building:** Add Gaussian contributions for each atom
4. **FFT:** Transform real-space density to reciprocal space
5. **Extraction:** Sample the reciprocal grid at HKL positions with symmetry averaging

## Verification

All approaches should give identical (or nearly identical) results. The correlation matrix should show 1.0 between all methods, and mean values should match.

**Note:** Small numerical differences may arise from:
- Floating point precision
- Order of operations in symmetry averaging
- Grid interpolation effects

In [7]:
# Compare all four calculation methods
# Stack the absolute values (amplitudes) and compute correlation matrix
methods = torch.stack([
    f_calc_automatic.abs(), 
    f_calc_semi_automatic.abs(), 
    f_calc_a_bit_less_automatic.abs(), 
    f_calc_manual.abs()
])

print("Correlation matrix (should be all 1.0):")
print(torch.corrcoef(methods))

print("\nMean |F_calc| for each method:")
print(f"  Automatic:           {f_calc_automatic.abs().mean().item():.6f}")
print(f"  Semi-automatic:      {f_calc_semi_automatic.abs().mean().item():.6f}")  
print(f"  Intermediate:        {f_calc_a_bit_less_automatic.abs().mean().item():.6f}")
print(f"  Manual:              {f_calc_manual.abs().mean().item():.6f}")

Correlation matrix (should be all 1.0):
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<ClampBackward1>)

Mean |F_calc| for each method:
  Automatic:           904.201172
  Semi-automatic:      904.201172
  Intermediate:        904.201172
  Manual:              827.251953


## Summary: Choosing the Right Approach

| Approach | Use Case | Complexity |
|----------|----------|------------|
| **Automatic** (`m(hkl)`) | Standard refinement, quick calculations | Simplest |
| **Semi-automatic** (`FFT` class) | Need density map access, multiple HKL sets, change cell spacegroup | Moderate |
| **Manual** (base functions) | Custom workflows, debugging, research | Most complex |

**Tips:**
- Start with the automatic approach and only go deeper if you need more control
- All approaches are differentiable - gradients flow through the entire pipeline
- The `apply_symmetry` parameter controls whether symmetry is applied during density building or SF extraction